# Lab 3.3 - Amazon SageMaker: Encoding Categorical Data

**Educate edition.** Replaces `en_us/3_3-machinelearning.ipynb`.

## Objectives
* Encode **ordinal** categorical data (categories with a meaningful order)
* Encode **non-ordinal / nominal** categorical data (no order)

**Cost note:** no AWS services used beyond the notebook instance.

This lab is a self-contained detour: it uses the UCI **Automobile**
dataset, which is rich in categorical features. It does not depend on
the vertebral column data.

In [ ]:
import warnings; warnings.simplefilter('ignore')
import pandas as pd, numpy as np
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

pd.set_option('display.max_columns', 40)

## Step 1 - Load the automobile dataset

This file has **no header row**, and missing values are encoded as the
literal string `?`. We supply the column names ourselves and tell
pandas to treat `?` as NaN.

In [ ]:
URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data'

COLS = ['symboling','normalized_losses','make','fuel_type','aspiration',
        'num_of_doors','body_style','drive_wheels','engine_location',
        'wheel_base','length','width','height','curb_weight','engine_type',
        'num_of_cylinders','engine_size','fuel_system','bore','stroke',
        'compression_ratio','horsepower','peak_rpm','city_mpg',
        'highway_mpg','price']

df = pd.read_csv(URL, header=None, names=COLS, na_values='?')
print('Shape:', df.shape)
df.head()

## Step 2 - Identify which columns are categorical

Columns loaded as `object` are the text/categorical ones. Note that
several numeric columns (`price`, `horsepower`, `bore`, ...) would also
have come in as `object` because of the `?` values - `na_values='?'`
keeps them numeric.

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(exclude='object').columns.tolist()

print('Categorical columns:', cat_cols)
print()
print('Numeric columns:', num_cols)
print()
print('Unique values per categorical column:')
for c in cat_cols:
    print(f'  {c:20s} {df[c].nunique():3d}  {sorted(df[c].dropna().unique())[:6]}')

## Step 3 - Handle missing values first

Encoders cannot process NaN. We fill categorical NaN with the column's
mode (most frequent value) and numeric NaN with the median.

In [ ]:
print('Missing before:')
print(df.isnull().sum()[df.isnull().sum() > 0])

for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

print()
print('Total missing after:', df.isnull().sum().sum())

## Step 4 - Encoding ORDINAL data

An **ordinal** feature has categories with a natural order. Encoding it
as integers is correct *and* useful, because the integer ordering
carries real information the model can use.

Two ordinal columns here:

* `num_of_doors`: `two` < `four`
* `num_of_cylinders`: `two` < `three` < `four` < `five` < `six` < `eight` < `twelve`

The critical part is **supplying the order explicitly**. If you let the
encoder pick, it sorts alphabetically - which would put `eight` before
`five` and destroy the meaning.

In [ ]:
door_order = ['two', 'four']
cyl_order  = ['two', 'three', 'four', 'five', 'six', 'eight', 'twelve']

ord_enc = OrdinalEncoder(categories=[door_order, cyl_order])
df[['num_of_doors_enc', 'num_of_cylinders_enc']] = ord_enc.fit_transform(
    df[['num_of_doors', 'num_of_cylinders']]
).astype(int)

df[['num_of_doors','num_of_doors_enc',
    'num_of_cylinders','num_of_cylinders_enc']].drop_duplicates().sort_values(
    'num_of_cylinders_enc')

### Why the explicit order matters

Compare what happens with the default alphabetical ordering. The
mapping below is wrong: it claims `eight` (0) is fewer cylinders than
`five` (1).

In [ ]:
bad = OrdinalEncoder()
bad.fit(df[['num_of_cylinders']])
print('Alphabetical (WRONG) mapping:')
for i, v in enumerate(bad.categories_[0]):
    print(f'  {v:8s} -> {i}')

print()
print('Explicit (CORRECT) mapping:')
for i, v in enumerate(cyl_order):
    print(f'  {v:8s} -> {i}')

## Step 5 - Encoding NON-ORDINAL (nominal) data

A **nominal** feature has no order. `body_style` values -
`convertible`, `hatchback`, `sedan`, `wagon`, `hardtop` - cannot be
ranked. Encoding them as 0-4 would make the model believe
`wagon` (4) is "greater than" `sedan` (3), which is meaningless and
will mislead distance- and linear-based algorithms.

The correct approach is **one-hot encoding**: one new binary column per
category.

In [ ]:
nominal = ['body_style', 'drive_wheels', 'fuel_type', 'aspiration',
           'engine_location']

try:
    oh = OneHotEncoder(sparse_output=False, dtype=int)
except TypeError:
    oh = OneHotEncoder(sparse=False, dtype=int)   # older scikit-learn

encoded = oh.fit_transform(df[nominal])
oh_df = pd.DataFrame(encoded, columns=oh.get_feature_names_out(nominal),
                     index=df.index)

print('Original columns:', len(nominal))
print('One-hot columns :', oh_df.shape[1])
oh_df.head()

### The same thing with pandas

`pd.get_dummies` is the one-line pandas equivalent, and is often more
convenient inside a notebook.

`drop_first=True` drops one category per feature. This avoids the
**dummy variable trap** (perfect multicollinearity) which matters for
linear models. Tree models like XGBoost do not care either way.

In [ ]:
dummies = pd.get_dummies(df[nominal], drop_first=True).astype(int)
print('With drop_first=True ->', dummies.shape[1], 'columns')
dummies.head()

### High-cardinality nominal features

`make` has 22 distinct values. One-hot encoding it adds 22 columns to a
205-row dataset - more columns than is healthy for the sample size.

Options: group rare categories into `Other`, use target/frequency
encoding, or let a tree model handle it natively. Here we group.

In [ ]:
print('make has', df['make'].nunique(), 'distinct values')
print(df['make'].value_counts().tail(10))

top = df['make'].value_counts().nlargest(8).index
df['make_grouped'] = np.where(df['make'].isin(top), df['make'], 'Other')

print()
print('After grouping ->', df['make_grouped'].nunique(), 'values')
print(df['make_grouped'].value_counts())

## Step 6 - Assemble the fully encoded dataset

Combine the numeric columns, the ordinal encodings and the one-hot
columns into a single model-ready frame.

In [ ]:
model_df = pd.concat([
    df[num_cols],
    df[['num_of_doors_enc', 'num_of_cylinders_enc']],
    pd.get_dummies(df[nominal + ['make_grouped']], drop_first=True).astype(int),
], axis=1)

print('Final shape:', model_df.shape)
print('All numeric:', model_df.select_dtypes(include='object').empty)
model_df.head()

## Summary: which encoder when?

| Situation | Technique | Why |
|---|---|---|
| Ordered categories (`small` < `medium` < `large`) | `OrdinalEncoder` **with explicit `categories=`** | Integer order carries real meaning |
| Unordered categories, few values | `OneHotEncoder` / `pd.get_dummies` | Avoids inventing a false ranking |
| Unordered, many values | Group rare into `Other`, or target/frequency encoding | One-hot would explode dimensionality |
| Binary target label | `LabelEncoder` | Maps two classes to 0/1 |

## Conclusion

You have:
* Encoded ordinal categorical data, controlling the category order
* Encoded non-ordinal categorical data with one-hot encoding
* Seen why the wrong choice injects false information into a model

**Remember to Stop the notebook instance when you are done.**

Next: `3_4-machinelearning.ipynb`